# IOAI — 2025 National Selection Weak Label Cv (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-national-selection-weak-label-cv/data.zip', 'd.zip')
    with zipfile.ZipFile('d.zip') as z: z.extractall('data')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Weak-Label CV — 모범답안 (캡션 약한라벨 + 처음부터 CNN)

**① 약한 라벨 유도**: 각 캡션에 8개 클래스 키워드가 하나만 등장하면 그 클래스로 라벨링. **② 처음부터 CNN**
학습(사전학습 없음): 128×128 → 소형 conv 네트워크 → 8클래스. **③ test 예측** → `submission.json`.
accuracy ≈ 0.5~0.7 (상수 베이스라인 0.12 대비).

In [ ]:
import pandas as pd, json, os, glob
captions = pd.read_csv("data/captions.csv")      # path, caption (약한 텍스트 라벨)
test = pd.read_csv("data/test_imgs.csv")         # path (라벨 없음)
print("caption imgs", len(captions), "| test imgs", len(test))

In [ ]:
import re, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["airplane","car","cat","dog","flower","fruit","motorbike","person"]
CID = {c:i for i,c in enumerate(CLASSES)}
KW = {"airplane":["airplane","aeroplane","plane","aircraft","jet","airliner"],"car":["car","automobile","sedan","truck","vehicle","suv"],
 "cat":["cat","kitten","kitty"],"dog":["dog","puppy","canine"],"flower":["flower","rose","tulip","daisy","blossom","petal"],
 "fruit":["fruit","apple","banana","orange","strawberr","grape","mango","pear"],"motorbike":["motorbike","motorcycle","scooter","moped"],
 "person":["person","man","woman","boy","girl","people","child","player","men","women"]}
def weak_label(cap):
    c = cap.lower(); h = [cl for cl,kws in KW.items() if any(re.search(r"\b"+re.escape(k), c) for k in kws)]
    return h[0] if len(h)==1 else None

captions["wl"] = captions.caption.apply(weak_label)
lab = captions[captions.wl.notna()].reset_index(drop=True)
print("weak-labeled train:", len(lab))

def load_img(path):
    img = Image.open(os.path.join("data", path)).convert("RGB").resize((128,128))
    return (np.asarray(img, np.float32)/255.0).transpose(2,0,1)

Xtr = torch.tensor(np.stack([load_img(p) for p in lab.path]))
ytr = torch.tensor([CID[c] for c in lab.wl])
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3,32,3,2,1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,64,3,2,1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,128,3,2,1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Linear(128, 8)
    def forward(self, x): return self.h(self.f(x))

net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3); lossf = nn.CrossEntropyLoss()
for ep in range(15):
    net.train()
    for xb, yb in loader:
        opt.zero_grad(); loss = lossf(net(xb.to(dev)), yb.to(dev)); loss.backward(); opt.step()
print("trained CNN, final loss", round(float(loss), 4))

net.eval(); preds = {}
with torch.no_grad():
    for p in test["path"]:
        x = torch.tensor(load_img(p)).unsqueeze(0).to(dev)
        preds[p] = CLASSES[int(net(x).argmax(1))]
with open("submission.json", "w") as f:
    json.dump(preds, f, indent=4)
print("saved submission.json", len(preds))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)